# IndoBERT Sentiment Training (Penginapan)
Notebook ini diformat ulang menjadi versi ringkas tanpa pola keputusan, memuat seluruh _pipeline_ IndoBERT dari awal hingga akhir (_End-to-End_).

**Tahapan:**
1. _Loading_ Data Terlabel (9.138 ulasan)
2. Tokenisasi IndoBERT (`indobenchmark/indobert-base-p1`)
3. _Training_ dengan `Trainer` HuggingFace (menggunakan `eval_strategy` dan _class weights_)
4. Inferensi dan Bayesian Shrinkage
5. Gabung (_Merge_) ke _Primary Dataset_


In [ ]:
# Install dependencies (jika di Kaggle, un-comment baris di bawah)
# !pip install transformers datasets evaluate accelerate scikit-learn numpy pandas torch

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
import evaluate

# Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

# Load Dataset (Ganti path jika di Kaggle: "/kaggle/input/.../PENGINAPAN_FASE_A_LABELLED_2026-06-14.csv")
DATA_PATH = r"D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_FASE_A_LABELLED_2026-06-14.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

# Mapping Label ke Angka (Positif: 0, Negatif: 1, Netral: 2)
# Sesuaikan dengan mapping yang sudah ada di label data Anda
label_map = {"Positif": 0, "Negatif": 1, "Netral": 2}
df['label_num'] = df['sentiment_label'].map(label_map)

# Split data: 80% Train, 20% Test
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

# Konversi ke format Dataset HuggingFace
dataset_train = Dataset.from_pandas(df_train[['cleaned_text', 'label_num']].rename(columns={'label_num': 'label'}))
dataset_test = Dataset.from_pandas(df_test[['cleaned_text', 'label_num']].rename(columns={'label_num': 'label'}))

print(f"Data latih: {len(dataset_train)} baris")
print(f"Data uji: {len(dataset_test)} baris")


In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], padding="max_length", truncation=True, max_length=128)

print("Memulai tokenisasi...")
tokenized_train = dataset_train.map(tokenize_function, batched=True)
tokenized_test = dataset_test.map(tokenize_function, batched=True)
print("Tokenisasi selesai!")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)


In [ ]:
# Hitung bobot kelas secara matematis (karena data kita imbalanced)
from torch import nn

class_weights = compute_class_weight('balanced', classes=np.unique(df_train['label_num']), y=df_train['label_num'])
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights: {class_weights_tensor}")

# Custom Trainer untuk menyuntikkan class weights
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Metrik Evaluasi
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",  # Menggunakan eval_strategy sesuai versi terbaru
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Memulai Training IndoBERT...")
trainer.train()

# Simpan model jika diperlukan
# trainer.save_model("./indobert-sentimen-penginapan")


In [ ]:
import torch.nn.functional as F

print("Melakukan inferensi prediksi ke semua 9.138 ulasan untuk mencari skor probabilitas...")

def predict_score(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probas = F.softmax(logits, dim=-1).cpu().numpy()[0]
        
    # Asumsi indeks: 0 = Positif, 1 = Negatif, 2 = Netral (Sesuai label_map)
    pos_val = probas[0]
    neg_val = probas[1]
    
    # Skor dari -1 (Sangat Negatif) hingga +1 (Sangat Positif)
    score = pos_val - neg_val
    confidence = np.max(probas)
    return score, confidence

# Karena prediksi satu-satu bisa lambat, batasi ke beberapa sampel dulu jika di lokal tanpa GPU
# Jika di Kaggle dengan GPU, Anda bisa prediksi keseluruhan.
# Untuk demo, mari iterasi dan simpan hasilnya
scores = []
confidences = []
for idx, text in enumerate(df['cleaned_text']):
    s, c = predict_score(text)
    scores.append(s)
    confidences.append(c)
    if (idx + 1) % 1000 == 0:
        print(f"Prediksi {idx + 1}/{len(df)} selesai.")

df['review_sentiment_score'] = scores
df['review_confidence'] = confidences

print("\n-- BAYESIAN SHRINKAGE --")
K_SHRINKAGE = 50.0
global_mean_score = np.mean(scores)
print(f"Global Mean Score: {global_mean_score:.4f}")

df_agg = df.groupby('target_penginapan_id').agg(
    hotel_review_count_analyzed=('reviewId', 'count'),
    hotel_sentiment_score=('review_sentiment_score', 'mean'),
    hotel_sentiment_confidence_mean=('review_confidence', 'mean')
).reset_index()

df_agg['hotel_adjusted_sentiment_score'] = (
    (df_agg['hotel_review_count_analyzed'] * df_agg['hotel_sentiment_score'] + K_SHRINKAGE * global_mean_score) 
    / (df_agg['hotel_review_count_analyzed'] + K_SHRINKAGE)
)

def get_sentiment_label(score):
    if score >= 0.6: return "Sangat Positif"
    if score >= 0.2: return "Positif"
    if score >= -0.2: return "Netral"
    if score >= -0.6: return "Negatif"
    return "Sangat Negatif"

df_agg['hotel_adjusted_sentiment_label'] = df_agg['hotel_adjusted_sentiment_score'].apply(get_sentiment_label)
print(f"Berhasil mengagregasi sentimen untuk {len(df_agg)} penginapan unik.")


In [ ]:
PRIMARY_PATH = r"D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_PRIMARY_DATA_FOCUS_CANDIDATE_SENTIMENT_UPDATED_2026-06-14.csv"
df_primary = pd.read_csv(PRIMARY_PATH, low_memory=False)

# Lakukan Merge (Left Join) berdasarkan id target_penginapan_id = placeId
df_final = pd.merge(df_primary, df_agg, left_on='placeId', right_on='target_penginapan_id', how='left')

coverage_akhir = df_final['hotel_adjusted_sentiment_score'].notna().sum()
total_rows = len(df_final)

print(f"Dataset Primer Total: {total_rows} baris")
print(f"Penginapan yang sudah memiliki skor sentimen AI IndoBERT: {coverage_akhir} properties")

# Export
EXPORT_PATH = r"D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_PRIMARY_FINAL_INDOBERT.csv"
df_final.to_csv(EXPORT_PATH, index=False)
print(f"Tersimpan di: {EXPORT_PATH}")
